In [1]:
!pip install pyspark

In [2]:
import os

dest_dir = "/content/data"
if not os.path.exists(dest_dir):
    os.makedirs(dest_dir)

# Set a new, temporary cache directory to bypass previous cache
os.environ["KAGGLEHUB_CACHE"] = dest_dir

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ssssws/chocolate-sales-dataset-2023-2024", force_download=True)


print("Path to dataset files:", path)

100%|██████████| 23.3M/23.3M [00:00<00:00, 151MB/s]

Extracting files...


Path to dataset files: /content/data/datasets/ssssws/chocolate-sales-dataset-2023-2024/versions/2


In [4]:
# Initiate PySpark session on Colab
from pyspark.sql import SparkSession

# Create a Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_PySpark_Test") \
    .getOrCreate()

# Check to see if it worked
print(spark)

In [5]:
sales_csv = path + "/sales.csv"
# Example: Read a CSV file into a PySpark DataFrame (replace 'your_dataset.csv' with the actual filename)
df = spark.read.csv(sales_csv, header=True, inferSchema=True)

# Verify by showing the schema and first few rows
df.printSchema()
df.show()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- profit: double (nullable = true)

+-----------+----------+----------+--------+-----------+--------+----------+--------+-------+-----+------+
|   order_id|order_date|product_id|store_id|customer_id|quantity|unit_price|discount|revenue| cost|profit|
+-----------+----------+----------+--------+-----------+--------+----------+--------+-------+-----+------+
|0RD00000001|2023-01-07|     P0080|    S093|    C040749|       5|     14.43|    0.15|  61.33|42.77| 18.56|
|0RD00000002|2023-10-22|     P0173|    S065|    C020161|       3|     12.01|     0.0|  36.03|19.06| 16.97|
|0RD00000003|2023-

In [6]:
from pyspark.sql.functions import col, count, when

# Check for missing values

df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()


+--------+----------+----------+--------+-----------+--------+----------+--------+-------+----+------+
|order_id|order_date|product_id|store_id|customer_id|quantity|unit_price|discount|revenue|cost|profit|
+--------+----------+----------+--------+-----------+--------+----------+--------+-------+----+------+
|       0|         0|         0|       0|          0|       0|         0|       0|      0|   0|     0|
+--------+----------+----------+--------+-----------+--------+----------+--------+-------+----+------+



In [7]:
# GroupBy Product
by_product = df.groupby('product_id').sum()
by_product.show()

+----------+-------------+------------------+------------------+------------------+-----------------+------------------+
|product_id|sum(quantity)|   sum(unit_price)|     sum(discount)|      sum(revenue)|        sum(cost)|       sum(profit)|
+----------+-------------+------------------+------------------+------------------+-----------------+------------------+
|     P0092|        14770|44240.090000000004| 269.5000000000007|126153.42999999998|75597.46999999988| 50555.78000000003|
|     P0192|        14685| 44170.16000000001| 265.6000000000005| 124985.2399999999|74986.55000000006| 49999.00000000005|
|     P0112|        14915|44773.750000000015| 280.2500000000007|127286.05000000005| 76610.5899999999|50675.360000000015|
|     P0122|        15065|          45099.87|276.65000000000083|128506.90000000002|77119.30999999994| 51387.69999999997|
|     P0167|        14472|          43153.81| 281.0500000000004|121854.17999999992|73308.33999999994| 48545.76999999995|
|     P0144|        14509| 43657

In [12]:
by_product.write.mode("overwrite").parquet("byproduct.parquet")

In [25]:
# Read in the Parquet file created above.
# Parquet files are self-describing so the schema is preserved.
# The result of loading a parquet file is also a DataFrame.
parquetFile = spark.read.parquet("byproduct.parquet")

# Parquet files can also be used to create a temporary view and then used in SQL statements.
parquetFile.createOrReplaceTempView("parquetFile")
profitable = spark.sql("SELECT product_id, `sum(profit)` as total_profit FROM parquetFile WHERE `sum(profit)` >= 40000")
profitable.show()

+----------+------------------+
|product_id|      total_profit|
+----------+------------------+
|     P0092| 50555.78000000003|
|     P0192| 49999.00000000005|
|     P0112|50675.360000000015|
|     P0122| 51387.69999999997|
|     P0167| 48545.76999999995|
|     P0144|49892.409999999974|
|     P0124| 50750.53999999998|
|     P0176| 49603.26000000004|
|     P0155|50778.470000000016|
|     P0172| 49842.95999999999|
|     P0085| 49973.39000000003|
|     P0118| 50208.54000000004|
|     P0009|50711.749999999956|
|     P0142|50144.509999999966|
|     P0027|  50651.3900000001|
|     P0088|  50748.6299999999|
|     P0169| 51126.99999999998|
|     P0076| 50676.63000000001|
|     P0021| 51298.91000000001|
|     P0015| 50670.47999999998|
+----------+------------------+
only showing top 20 rows


In [54]:
from urllib.request import urlretrieve

# Load a larger dataset of land sattelite from the UCI repository
# Description: https://archive.ics.uci.edu/dataset/146/statlog+landsat+satellite
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.trn"

#sat_data = pd.read_csv(url, header=None, sep="\\s+")
sat_csv = "sat.csv"
sat_data = urlretrieve(url, sat_csv)

In [55]:
sat_df = spark.read.csv(sat_csv, header=None, sep=' ', inferSchema=True, ignoreLeadingWhiteSpace=True,
    ignoreTrailingWhiteSpace=True)

# Verify by showing the schema and first few rows
sat_df.printSchema()
sat_df.show()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: integer (nullable = true)
 |-- _c4: integer (nullable = true)
 |-- _c5: integer (nullable = true)
 |-- _c6: integer (nullable = true)
 |-- _c7: integer (nullable = true)
 |-- _c8: integer (nullable = true)
 |-- _c9: integer (nullable = true)
 |-- _c10: integer (nullable = true)
 |-- _c11: integer (nullable = true)
 |-- _c12: integer (nullable = true)
 |-- _c13: integer (nullable = true)
 |-- _c14: integer (nullable = true)
 |-- _c15: integer (nullable = true)
 |-- _c16: integer (nullable = true)
 |-- _c17: integer (nullable = true)
 |-- _c18: integer (nullable = true)
 |-- _c19: integer (nullable = true)
 |-- _c20: integer (nullable = true)
 |-- _c21: integer (nullable = true)
 |-- _c22: integer (nullable = true)
 |-- _c23: integer (nullable = true)
 |-- _c24: integer (nullable = true)
 |-- _c25: integer (nullable = true)
 |-- _c26: integer (nullable = true)
 |-- _

In [48]:
sat_df.summary().show()

+-------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+-----------------+
|summary|               _c0|               _c1|               _c2|               _c3|               _c4|              _c5|               _c6|               _c7|               _c8|               _c9|              _c10|              _c11|              _c12|              _c13|              _c14| 

In [60]:
# Extract labels
sat_last_col = sat_df.columns[-1]
sat_labels = sat_df.select(sat_last_col).distinct()
sat_labels.show()

+----+
|_c36|
+----+
|   1|
|   3|
|   5|
|   4|
|   7|
|   2|
+----+



In [63]:
# GroupBy label
by_label = sat_df.groupby(sat_last_col).mean()
by_label.show()

+----+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+---------+
|_c36|          avg(_c0)|          avg(_c1)|          avg(_c2)|          avg(_c3)|         avg(_c4)|          avg(_c5)|          avg(_c6)|          avg(_c7)|          avg(_c8)|          avg(_c9)|         avg(_c10)|         avg(_c11)|        avg(_c12)|         avg(_c13)|         avg(_c14)|         avg(_